In [ ]:
%pip install pyspark pandas matplotlib

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("matches.csv")

if not DATA_PATH.exists():
    candidate = Path("data") / "matches.csv"

    if candidate.exists():
        DATA_PATH = candidate

print("Input file:", DATA_PATH.resolve())

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# 1. Install Java and locate the installation directory dynamically
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

import os
import sys
import subprocess

# Dynamically discover the correct Java directory path
try:
    java_path = subprocess.check_output("readlink -f /usr/bin/java", shell=True).decode("utf-8").strip()
    # Exclude "/bin/java" suffix to get the root JAVA_HOME
    if "/bin/java" in java_path:
        os.environ["JAVA_HOME"] = java_path.split("/bin/java")[0]
except Exception:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Ensure Java binaries are available in the PATH
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# Clear PySpark environment variables pointing to obsolete installations
os.environ.pop("SPARK_HOME", None)

# Import PySpark with clean environment state
import pyspark
from pyspark.sql import SparkSession

# Initialize the Spark Session cleanly
spark = (
    SparkSession.builder
    .appName("IPLMatchPredictionAnalysis")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

# 5. Locate matches dataset and load cleanly
from pathlib import Path
data_file = Path("/content/matches.csv")
if not data_file.exists():
    data_file = Path("/content/matches (1).csv")

ipl = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(data_file))
)

print("Total Rows Loaded:", ipl.count())
ipl.printSchema()
ipl.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F

required = {
    "team1",
    "team2",
    "venue",
    "toss_winner",
    "toss_decision",
    "winner"
}

missing = required - set(ipl.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}"
    )

clean = (
    ipl
    .dropDuplicates()
    .filter(
        (F.col("team1").isNotNull()) &
        (F.col("team2").isNotNull()) &
        (F.col("venue").isNotNull()) &
        (F.col("toss_winner").isNotNull()) &
        (F.col("toss_decision").isNotNull()) &
        (F.col("winner").isNotNull())
    )
)

print("Original Rows:", ipl.count())
print("Valid Rows:", clean.count())
print(
    "Dropped Rows:",
    ipl.count() - clean.count()
)

clean.show(10, truncate=False)

In [ ]:
clean.select(

    F.count("*").alias("total_matches"),

    F.countDistinct("team1")
    .alias("unique_team1"),

    F.countDistinct("team2")
    .alias("unique_team2"),

    F.countDistinct("venue")
    .alias("total_venues"),

    F.countDistinct("winner")
    .alias("winning_teams")

).show(truncate=False)

In [ ]:
team_summary = (
    clean
    .groupBy("winner")
    .agg(
        F.count("*").alias("total_wins")
    )
    .orderBy(
        F.desc("total_wins")
    )
)

team_summary.show(
    100,
    truncate=False
)

In [ ]:
venue_summary = (
    clean
    .groupBy("venue")
    .agg(
        F.count("*")
        .alias("total_matches")
    )
    .orderBy(
        F.desc("total_matches")
    )
)

venue_summary.show(
    100,
    truncate=False
)

In [ ]:
toss_summary = (
    clean
    .groupBy("toss_decision")
    .agg(
        F.count("*")
        .alias("total_matches")
    )
    .orderBy(
        F.desc("total_matches")
    )
)

toss_summary.show(
    truncate=False
)

In [ ]:
total_matches = clean.count()

toss_match_wins = (
    clean
    .filter(
        F.col("toss_winner")
        == F.col("winner")
    )
    .count()
)

percentage = (
    toss_match_wins /
    total_matches
) * 100

print(
    "Total Matches:",
    total_matches
)

print(
    "Toss Winner Also Won Match:",
    toss_match_wins
)

# Explicitly use Python's built-in round function to bypass PySpark's legacy NumPy 2.0 incompatibilities
print(
    "Toss Impact Percentage:",
    __builtins__.round(percentage, 2),
    "%"
)

In [ ]:
team_pd = (
    team_summary
    .limit(15)
    .toPandas()
)

plt.figure(
    figsize=(11, 5)
)

plt.bar(
    team_pd["winner"],
    team_pd["total_wins"]
)

plt.title(
    "IPL Team-wise Match Wins"
)

plt.xlabel(
    "IPL Team"
)

plt.ylabel(
    "Number of Wins"
)

plt.xticks(
    rotation=60
)

plt.tight_layout()

plt.show()

In [ ]:
venue_pd = (
    venue_summary
    .limit(15)
    .toPandas()
)

venue_pd = venue_pd.sort_values(
    "total_matches"
)

plt.figure(
    figsize=(10, 7)
)

plt.barh(
    venue_pd["venue"],
    venue_pd["total_matches"]
)

plt.title(
    "IPL Matches by Venue"
)

plt.xlabel(
    "Number of Matches"
)

plt.ylabel(
    "Venue"
)

plt.tight_layout()

plt.show()

In [ ]:
OUTPUT_PATH = "output/ipl_team_analysis"

team_summary.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(OUTPUT_PATH)

print(
    "IPL results written to:",
    OUTPUT_PATH
)

In [ ]:
VENUE_OUTPUT = "output/ipl_venue_analysis"

venue_summary.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(VENUE_OUTPUT)

print(
    "Venue results written to:",
    VENUE_OUTPUT
)

In [ ]:
!hdfs dfs -mkdir -p /ipl/matches

# Safely upload the dataset file depending on which file exists in your active directory
!if [ -f "/content/matches.csv" ]; then hdfs dfs -put -f /content/matches.csv /ipl/matches/; else hdfs dfs -put -f "/content/matches (1).csv" /ipl/matches/matches.csv; fi

!hdfs dfs -ls /ipl/matches

In [ ]:
!hdfs dfs -cat /ipl/matches/matches.csv | head

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IPL Match Analysis") \
    .getOrCreate()

hdfs_ipl = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/content/matches.csv")
)

print("Dataset loaded successfully!")

hdfs_ipl.show(10, truncate=False)
hdfs_ipl.printSchema()

In [ ]:
%pip install cassandra-driver

In [ ]:
from cassandra.cluster import Cluster

try:
    # Connect to the local Cassandra instance
    cluster = Cluster(['127.0.0.1'])
    session = cluster.connect()

    # Execute the Keyspace creation statement as a Python string
    session.execute("""
        CREATE KEYSPACE IF NOT EXISTS ipl_analysis
        WITH replication = {
            'class': 'SimpleStrategy',
            'replication_factor': 1
        };
    """)

    # Switch to the new keyspace
    session.set_keyspace('ipl_analysis')
    print("Keyspace 'ipl_analysis' created and selected successfully.")

except Exception as e:
    print("Error connecting to Cassandra or running queries:", e)
    print("Please ensure your Cassandra database service is running.")

In [ ]:
from cassandra.cluster import Cluster

try:
    # Connect to the local Cassandra instance and connect to the keyspace
    cluster = Cluster(['127.0.0.1'])
    session = cluster.connect('ipl_analysis')

    # Execute the table creation statement inside a Python string
    session.execute("""
        CREATE TABLE IF NOT EXISTS team_performance (
            team TEXT PRIMARY KEY,
            total_wins INT
        );
    """)
    print("Table 'team_performance' created successfully inside 'ipl_analysis' keyspace.")

except Exception as e:
    print("Error creating table:", e)
    print("Ensure that your local Cassandra service is running and keyspace 'ipl_analysis' exists.")

In [ ]:
from cassandra.cluster import Cluster

try:
    # Connect to the local Cassandra instance
    cluster = Cluster(["127.0.0.1"])
    session = cluster.connect()

    # Create Keyspace
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS ipl_analysis
    WITH replication = {
        'class': 'SimpleStrategy',
        'replication_factor': 1
    }
    """)

    # Use Keyspace
    session.set_keyspace("ipl_analysis")

    # Create Table
    session.execute("""
    CREATE TABLE IF NOT EXISTS team_performance (
        team TEXT PRIMARY KEY,
        total_wins INT
    )
    """)

    print("Cassandra table created successfully")

except Exception as e:
    print("Error connecting to Cassandra or executing queries:", e)
    print("Please ensure your Cassandra database server is installed and running.")

In [ ]:
team_data = team_summary.collect()

try:
    for row in team_data:
        session.execute(
            """
            INSERT INTO team_performance
            (team, total_wins)
            VALUES (%s, %s)
            """,
            (
                row["winner"],
                row["total_wins"]
            )
        )
    print("IPL team results stored in Cassandra")
except NameError as ne:
    print("Error: The 'session' variable is not defined.")
    print("This happens because the previous cells failed to establish a connection to a running Cassandra server.")
except Exception as e:
    print("An error occurred while inserting data:", e)

In [ ]:
try:
    rows = session.execute(
        "SELECT * FROM team_performance"
    )

    for row in rows:
        print(
            row.team,
            row.total_wins
        )
except NameError:
    print("Error: The 'session' variable is not defined.")
    print("This is because the connection to your local Cassandra service failed to establish in the earlier setup cells.")
except Exception as e:
    print("An error occurred while executing the query:", e)

In [ ]:
spark.stop()

print(
    "Spark Session Stopped"
)

In [ ]:
!pip -q install streamlit plotly pandas

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="IPL Match Analysis",
    page_icon="🏏",
    layout="wide"
)

st.markdown("""
<style>

.stApp {
    background: linear-gradient(
        135deg,
        #f4f8ff,
        #ffffff
    );
}

h1 {
    color: #16325c;
}

h2, h3 {
    color: #176b87;
}

[data-testid="stMetric"] {
    background: white;
    padding: 18px;
    border-radius: 12px;
    box-shadow: 0 2px 8px #00000012;
}

</style>
""", unsafe_allow_html=True)


st.title(
    "🏏 IPL Match Analysis"
)

st.markdown(
    "### Big Data Analytics using PySpark"
)

st.divider()


uploaded_file = st.file_uploader(
    "Upload matches.csv",
    type=["csv"]
)


if uploaded_file is not None:

    df = pd.read_csv(
        uploaded_file
    )

    st.success(
        "Dataset uploaded successfully!"
    )

    # -------------------------
    # Metrics
    # -------------------------

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Total Matches",
        len(df)
    )

    col2.metric(
        "Total Teams",
        df["winner"]
        .nunique()
    )

    col3.metric(
        "Total Venues",
        df["venue"]
        .nunique()
    )


    # -------------------------
    # Dataset
    # -------------------------

    st.subheader(
        "IPL Dataset"
    )

    st.dataframe(
        df.head(20),
        use_container_width=True
    )


    # -------------------------
    # Team Wins
    # -------------------------

    st.subheader(
        "Team-wise Match Wins"
    )

    wins = (
        df["winner"]
        .value_counts()
        .reset_index()
    )

    wins.columns = [
        "Team",
        "Wins"
    ]

    fig = px.bar(
        wins,
        x="Team",
        y="Wins",
        title="IPL Team-wise Wins"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


    # -------------------------
    # Venue Analysis
    # -------------------------

    st.subheader(
        "Venue-wise IPL Matches"
    )

    venues = (
        df["venue"]
        .value_counts()
        .head(15)
        .reset_index()
    )

    venues.columns = [
        "Venue",
        "Matches"
    ]

    fig2 = px.bar(
        venues,
        x="Matches",
        y="Venue",
        orientation="h",
        title="Top IPL Venues"
    )

    st.plotly_chart(
        fig2,
        use_container_width=True
    )


    # -------------------------
    # Toss Analysis
    # -------------------------

    st.subheader(
        "Toss Decision Analysis"
    )

    toss = (
        df["toss_decision"]
        .value_counts()
        .reset_index()
    )

    toss.columns = [
        "Decision",
        "Count"
    ]

    fig3 = px.pie(
        toss,
        names="Decision",
        values="Count",
        title="IPL Toss Decisions"
    )

    st.plotly_chart(
        fig3,
        use_container_width=True
    )


    # -------------------------
    # Toss Impact
    # -------------------------

    toss_winner_match_winner = (
        df["toss_winner"]
        == df["winner"]
    ).sum()

    percentage = (
        toss_winner_match_winner
        / len(df)
    ) * 100

    st.subheader(
        "Toss Impact"
    )

    st.metric(
        "Toss Winner Also Won Match",
        f"{percentage:.2f}%"
    )

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px

# -------------------------------------------------
# PAGE CONFIGURATION
# -------------------------------------------------
st.set_page_config(
    page_title="IPL Analytics",
    page_icon="🏏",
    layout="wide"
)

# -------------------------------------------------
# CSS
# -------------------------------------------------
st.markdown("""
<style>
    .stApp {
        background: linear-gradient(135deg, #f5f7fa, #e8eef7);
    }

    .main-title {
        text-align: center;
        font-size: 45px;
        font-weight: 800;
        color: #153e75;
    }

    .subtitle {
        text-align: center;
        color: #555;
        font-size: 18px;
        margin-bottom: 25px;
    }

    [data-testid="stMetric"] {
        background-color: white;
        border-radius: 15px;
        padding: 20px;
        box-shadow: 0px 3px 10px rgba(0,0,0,0.08);
    }

    .prediction {
        background-color: white;
        padding: 25px;
        border-radius: 15px;
        text-align: center;
        font-size: 22px;
        font-weight: bold;
        box-shadow: 0px 3px 10px rgba(0,0,0,0.08);
    }
</style>
""", unsafe_allow_html=True)


# -------------------------------------------------
# TITLE
# -------------------------------------------------
st.markdown(
    '<div class="main-title">🏏 IPL Match Analytics Dashboard</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">Big Data Analytics using PySpark, Hadoop and Cassandra</div>',
    unsafe_allow_html=True
)

st.divider()


# -------------------------------------------------
# FILE UPLOAD
# -------------------------------------------------
uploaded_file = st.file_uploader(
    "📂 Upload IPL matches.csv",
    type=["csv"]
)


if uploaded_file is not None:

    df = pd.read_csv(uploaded_file)

    # Remove duplicates
    df = df.drop_duplicates()

    # Clean important columns
    required_columns = [
        "team1",
        "team2",
        "venue",
        "toss_winner",
        "toss_decision",
        "winner"
    ]

    existing = [x for x in required_columns if x in df.columns]

    df = df.dropna(subset=existing)

    st.success("✅ IPL dataset loaded successfully!")


    # -------------------------------------------------
    # SIDEBAR
    # -------------------------------------------------
    st.sidebar.title("🏏 IPL Dashboard")

    st.sidebar.write(
        "IPL Match Analysis & Prediction"
    )

    teams = sorted(df["winner"].dropna().unique())

    selected_team = st.sidebar.selectbox(
        "Select Team",
        ["All Teams"] + teams
    )

    if selected_team != "All Teams":

        filtered_df = df[
            (df["team1"] == selected_team) |
            (df["team2"] == selected_team)
        ]

    else:
        filtered_df = df


    # -------------------------------------------------
    # DASHBOARD METRICS
    # -------------------------------------------------
    st.subheader("📊 IPL Overview")

    c1, c2, c3, c4 = st.columns(4)

    c1.metric(
        "Total Matches",
        len(filtered_df)
    )

    c2.metric(
        "IPL Teams",
        df["winner"].nunique()
    )

    c3.metric(
        "Venues",
        df["venue"].nunique()
    )

    top_team = df["winner"].value_counts().index[0]

    c4.metric(
        "Most Successful Team",
        top_team
    )

    st.divider()


    # -------------------------------------------------
    # TEAM WINS
    # -------------------------------------------------
    st.subheader("🏆 Team-wise Match Wins")

    wins = (
        filtered_df["winner"]
        .value_counts()
        .reset_index()
    )

    wins.columns = [
        "Team",
        "Wins"
    ]

    fig = px.bar(
        wins,
        x="Team",
        y="Wins",
        title="IPL Team Performance",
        text="Wins"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


    # -------------------------------------------------
    # VENUE ANALYSIS
    # -------------------------------------------------
    st.subheader("🏟️ Venue-wise Match Analysis")

    venue_data = (
        filtered_df["venue"]
        .value_counts()
        .head(15)
        .reset_index()
    )

    venue_data.columns = [
        "Venue",
        "Matches"
    ]

    fig2 = px.bar(
        venue_data,
        x="Matches",
        y="Venue",
        orientation="h",
        title="Top IPL Venues"
    )

    st.plotly_chart(
        fig2,
        use_container_width=True
    )


    # -------------------------------------------------
    # TOSS ANALYSIS
    # -------------------------------------------------
    st.subheader("🪙 Toss Decision Analysis")

    toss_data = (
        filtered_df["toss_decision"]
        .value_counts()
        .reset_index()
    )

    toss_data.columns = [
        "Decision",
        "Count"
    ]

    fig3 = px.pie(
        toss_data,
        names="Decision",
        values="Count",
        title="Bat vs Field Decision"
    )

    st.plotly_chart(
        fig3,
        use_container_width=True
    )


    # -------------------------------------------------
    # TOSS IMPACT
    # -------------------------------------------------
    st.subheader("🎯 Toss Impact on Match")

    toss_wins = (
        filtered_df["toss_winner"]
        == filtered_df["winner"]
    ).sum()

    if len(filtered_df) > 0:

        toss_percentage = (
            toss_wins / len(filtered_df)
        ) * 100

    else:
        toss_percentage = 0

    c1, c2 = st.columns(2)

    c1.metric(
        "Toss Winner Won Match",
        toss_wins
    )

    c2.metric(
        "Toss Impact",
        f"{toss_percentage:.2f}%"
    )


    # -------------------------------------------------
    # SIMPLE MATCH PREDICTION
    # -------------------------------------------------
    st.divider()

    st.subheader("🔮 IPL Match Prediction")

    col1, col2 = st.columns(2)

    team1 = col1.selectbox(
        "Select Team 1",
        teams,
        key="team1"
    )

    team2_options = [
        x for x in teams
        if x != team1
    ]

    team2 = col2.selectbox(
        "Select Team 2",
        team2_options,
        key="team2"
    )


    if st.button(
        "🏏 Predict Winner",
        use_container_width=True
    ):

        team1_wins = (
            df["winner"] == team1
        ).sum()

        team2_wins = (
            df["winner"] == team2
        ).sum()

        total = team1_wins + team2_wins

        if total > 0:

            team1_probability = (
                team1_wins / total
            ) * 100

            team2_probability = (
                team2_wins / total
            ) * 100

            predicted_winner = (
                team1
                if team1_probability >= team2_probability
                else team2
            )

            st.markdown(
                f"""
                <div class="prediction">
                🏆 Predicted Winner:
                <br><br>
                {predicted_winner}
                </div>
                """,
                unsafe_allow_html=True
            )

            st.write("")

            p1, p2 = st.columns(2)

            p1.metric(
                team1,
                f"{team1_probability:.2f}%"
            )

            p2.metric(
                team2,
                f"{team2_probability:.2f}%"
            )


    # -------------------------------------------------
    # DATASET
    # -------------------------------------------------
    st.divider()

    st.subheader("📋 IPL Dataset")

    st.dataframe(
        filtered_df,
        use_container_width=True
    )


else:

    st.info(
        "👆 Upload your matches.csv file to open the IPL dashboard."
    )

In [ ]:
!streamlit run app.py \
--server.port 8501 \
--server.address 0.0.0.0 \
> /content/streamlit.log 2>&1 &

In [ ]:
pip install streamlit plotly pandas

In [ ]:
!streamlit run app.py

In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true

In [ ]:
import streamlit as st
import pandas as pd

@st.cache_data
def load_data(file):
    return pd.read_csv(file)

# Ensure uploaded_file is defined before using
try:
    df = load_data(uploaded_file)
except NameError:
    print("Error: 'uploaded_file' is not defined. Please run a file upload cell first.")

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="IPL Dashboard",
    page_icon="🏏",
    layout="wide"
)

# ---------- TITLE ----------
st.title("🏏 IPL Match Analysis Dashboard")
st.write("Big Data Analytics Project using IPL Dataset")

# ---------- CACHE ----------
@st.cache_data
def load_data(file):
    return pd.read_csv(file)

# ---------- UPLOAD ----------
uploaded_file = st.file_uploader(
    "📂 Upload matches.csv",
    type=["csv"]
)

if uploaded_file is not None:

    # NOW uploaded_file exists
    df = load_data(uploaded_file)

    df = df.drop_duplicates()

    required = [
        "team1",
        "team2",
        "venue",
        "toss_winner",
        "toss_decision",
        "winner"
    ]

    available = [c for c in required if c in df.columns]

    df = df.dropna(subset=available)

    st.success("✅ Dataset Loaded Successfully")

    # ---------- METRICS ----------
    col1, col2, col3, col4 = st.columns(4)

    col1.metric(
        "Total Matches",
        len(df)
    )

    col2.metric(
        "Teams",
        df["winner"].nunique()
    )

    col3.metric(
        "Venues",
        df["venue"].nunique()
    )

    top_team = df["winner"].value_counts().idxmax()

    col4.metric(
        "Top Team",
        top_team
    )

    # ---------- TEAM WINS ----------
    st.subheader("🏆 Team-wise Wins")

    wins = df["winner"].value_counts().reset_index()
    wins.columns = ["Team", "Wins"]

    fig1 = px.bar(
        wins,
        x="Team",
        y="Wins",
        text="Wins",
        title="IPL Team-wise Match Wins"
    )

    st.plotly_chart(
        fig1,
        use_container_width=True
    )

    # ---------- VENUE ----------
    st.subheader("🏟️ Venue Analysis")

    venues = df["venue"].value_counts().head(15).reset_index()
    venues.columns = ["Venue", "Matches"]

    fig2 = px.bar(
        venues,
        x="Matches",
        y="Venue",
        orientation="h",
        title="Top IPL Venues"
    )

    st.plotly_chart(
        fig2,
        use_container_width=True
    )

    # ---------- TOSS ----------
    st.subheader("🪙 Toss Decision Analysis")

    toss = df["toss_decision"].value_counts().reset_index()
    toss.columns = ["Decision", "Count"]

    fig3 = px.pie(
        toss,
        names="Decision",
        values="Count",
        title="Bat vs Field"
    )

    st.plotly_chart(
        fig3,
        use_container_width=True
    )

    # ---------- TOSS IMPACT ----------
    st.subheader("🎯 Toss Impact")

    toss_wins = (
        df["toss_winner"] == df["winner"]
    ).sum()

    percentage = (toss_wins / len(df)) * 100

    c1, c2 = st.columns(2)

    c1.metric(
        "Toss Winner Won Match",
        toss_wins
    )

    c2.metric(
        "Toss Impact",
        f"{percentage:.2f}%"
    )

    # ---------- PREDICTION ----------
    st.subheader("🔮 IPL Match Prediction")

    teams = sorted(
        set(df["team1"].dropna()) |
        set(df["team2"].dropna())
    )

    c1, c2 = st.columns(2)

    team1 = c1.selectbox(
        "Team 1",
        teams
    )

    team2_list = [
        t for t in teams
        if t != team1
    ]

    team2 = c2.selectbox(
        "Team 2",
        team2_list
    )

    if st.button("🏏 Predict Winner"):

        t1_wins = (df["winner"] == team1).sum()
        t2_wins = (df["winner"] == team2).sum()

        total = t1_wins + t2_wins

        if total > 0:

            p1 = (t1_wins / total) * 100
            p2 = (t2_wins / total) * 100

            winner = team1 if p1 >= p2 else team2

            st.success(
                f"🏆 Predicted Winner: {winner}"
            )

            a, b = st.columns(2)

            a.metric(
                team1,
                f"{p1:.2f}%"
            )

            b.metric(
                team2,
                f"{p2:.2f}%"
            )

    # ---------- DATA ----------
    st.subheader("📋 IPL Dataset")

    st.dataframe(
        df.head(100),
        use_container_width=True
    )

else:

    st.info(
        "👆 Please upload matches.csv to start the dashboard."
    )

In [ ]:
!curl -I http://localhost:8501

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=900
)

In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run app.py \
--server.port 8501 \
--server.address 0.0.0.0 \
--server.headless true \
--server.enableCORS false \
--server.enableXsrfProtection false \
> /content/streamlit.log 2>&1 &

In [ ]:
import time
time.sleep(5)

!curl -I http://localhost:8501

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=1000
)

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=1200,
    cache_in_notebook=False
)

In [ ]:
!tail -100 /content/streamlit.log

In [ ]:
!pkill -f streamlit


In [ ]:
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --browser.gatherUsageStats false \
  > /content/streamlit.log 2>&1 &

In [ ]:
import time
time.sleep(5)

!curl -s http://localhost:8501/_stcore/health

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=1000,
    cache_in_notebook=False
)


In [ ]:
import os
import subprocess
import re
import time

# Download cloudflared if it doesn't exist
if not os.path.exists("/content/cloudflared"):
    print("Downloading cloudflared...")
    subprocess.run(
        ["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/content/cloudflared"],
        check=True
    )
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)
    print("cloudflared installed successfully!")

process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Creating IPL Dashboard link...")

for line in process.stdout:
    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )

    if match:
        url = match.group(0)
        print("\n🏏 IPL DASHBOARD LINK:")
        print(url)
        break